In [1]:
import openpyxl
from collections import defaultdict

from pathlib import Path

In [2]:
XLSX = "../Daftar sampel perusahaan rutin.xlsx"
TEMPLATE = Path("../contoh_1_baris.sql").read_text()
PLACEHOLDER = "'CITRA OKU MEDIA , PT'"
LEVEL_CODE = "a.level_2_full_code = '1601'"

wb = openpyxl.load_workbook(XLSX, read_only=True, data_only=True)

In [ ]:
def esc(s):
    return s.strip().upper().replace("'", "''") if s else ""


CHUNK = 10
OUT_COLS = ", ".join(
    f"b.{c}"
    for c in [
        "assignment_id",
        "level_2_full_code",
        "nama_usaha_dicari",
        "nama_usaha",
        "nama_usaha_edit",
        "assignment_status_alias",
        "keberadaan_usaha_label",
        "nama_usaha_sim",
        "nama_usaha_edit_sim",
    ]
)

# 1. Kumpulkan semua nama usaha, dikelompokkan per kode_kab (lintas semua sheet)
data_per_kab = defaultdict(list)

for sheet in wb.sheetnames:
    rows = wb[sheet].iter_rows(values_only=True)
    header = next(rows)
    idx_nama = header.index("Nama Perusahaan")
    idx_kab = header.index("kode_kab")

    for r in rows:
        nama = r[idx_nama]
        kode_kab = r[idx_kab]
        if nama and str(nama).strip() and kode_kab:
            data_per_kab[str(kode_kab).strip()].append(nama)

# 2. Loop per kode_kab, bukan per sheet
for kode_kab, names in data_per_kab.items():
    for i in range(0, len(names), CHUNK):
        chunk = names[i : i + CHUNK]
        blocks = [
            "("
            + TEMPLATE.replace(PLACEHOLDER, f"'{esc(n)}'").replace(
                LEVEL_CODE, f"a.level_2_full_code = '{kode_kab}'"
            )
            + ")"
            for n in chunk
        ]
        out = f"{kode_kab}_{i // CHUNK + 1}.sql"
        Path(out).write_text(
            f"SELECT {OUT_COLS} FROM (\n{' UNION ALL '.join(blocks)}\n) b;"
        )

    n_files = (len(names) - 1) // CHUNK + 1
    print(
        f"{kode_kab}: {len(names)} rows -> {kode_kab}_1.sql ... {kode_kab}_{n_files}.sql"
    )

1601: 45 rows -> 1601_1.sql ... 1601_5.sql
1602: 71 rows -> 1602_1.sql ... 1602_8.sql
1603: 78 rows -> 1603_1.sql ... 1603_8.sql
1604: 78 rows -> 1604_1.sql ... 1604_8.sql
1605: 55 rows -> 1605_1.sql ... 1605_6.sql
1606: 127 rows -> 1606_1.sql ... 1606_13.sql
1607: 188 rows -> 1607_1.sql ... 1607_19.sql
1608: 19 rows -> 1608_1.sql ... 1608_2.sql
1609: 36 rows -> 1609_1.sql ... 1609_4.sql
1610: 62 rows -> 1610_1.sql ... 1610_7.sql
1611: 20 rows -> 1611_1.sql ... 1611_2.sql
1612: 19 rows -> 1612_1.sql ... 1612_2.sql
1613: 48 rows -> 1613_1.sql ... 1613_5.sql
1671: 207 rows -> 1671_1.sql ... 1671_21.sql
1672: 33 rows -> 1672_1.sql ... 1672_4.sql
1673: 16 rows -> 1673_1.sql ... 1673_2.sql
1674: 48 rows -> 1674_1.sql ... 1674_5.sql


: 